In [1]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

SCHEME = "LU-SG"
RESULTS_DIR = Path("./results")

def load_results(suffix: str) -> pl.DataFrame:
    dfs = []
    for f in RESULTS_DIR.iterdir():
        if not (f.is_file() and f.name.endswith(f"_{suffix}.csv")):
            continue
        parts = f.name.split("_")
        c_idx = next(i for i, p in enumerate(parts) if p.startswith("c") and p[1:].isdigit())
        scheme = "_".join(parts[:c_idx])
        if scheme != SCHEME:
            continue
        df = pl.read_csv(str(f))
        n_clients = int(parts[c_idx][1:])
        df = df.with_columns(pl.lit(scheme).alias("scheme"), pl.lit(n_clients).alias("nClients"))
        dfs.append(df)
    return pl.concat(dfs) if dfs else pl.DataFrame()

## Analytical Reference Values

- GPU: RTX 5060 modelled with 5 DVFS operating points.
- Workload: YOLOv8n (8.7 GFLOP, 60 KB input, 1 KB output).
- Topology: Point-to-point, 1 Gbps / 2 μs per link.
- Scheme: LU-SG

In [ ]:
PEAK_RATE = 19.2e12       # FP32 FLOPS
MIN_FREQ = 2.280e9       # Hz
MAX_FREQ = 2.50e9       # Hz
EFFICIENCY = 0.075
MEM_BW = 448e9            # bytes/s
TRANSFER_BW = 32e9        # bytes/s
STATIC_POWER = 9.0        # W
C_EFF = 1.94e-8           # F

OPP_BASE = (2.280e9, 0.880)  # (freq, voltage)

# Workload
COMPUTE_DEMAND = 8.7e9    # FLOP
INPUT_SIZE = 60_000       # bytes
OUTPUT_SIZE = 1_000       # bytes
FRAME_RATE = 30.0
SIM_TIME = 20.0
TOTAL_TIME = SIM_TIME + 2.0

RATE_AT_BASE = PEAK_RATE * (MIN_FREQ / MAX_FREQ)
EFF_RATE = RATE_AT_BASE * EFFICIENCY

def processing_time(eff_rate):
    return INPUT_SIZE / TRANSFER_BW + COMPUTE_DEMAND / eff_rate + OUTPUT_SIZE / TRANSFER_BW

T_PROC = processing_time(EFF_RATE)
T_COMPUTE = COMPUTE_DEMAND / EFF_RATE
def dynamic_power(opp):
    f, v = opp
    return C_EFF * v**2 * f

P_DYN = dynamic_power(OPP_BASE)

print(f"Effective rate:  {EFF_RATE:.4e} FLOPS")
print(f"Processing time: {T_PROC*1e3:.4f} ms")
print(f"Compute time:    {T_COMPUTE*1e3:.4f} ms")
print(f"P_dynamic:       {P_DYN:.2f} W")
print(f"P_static:        {STATIC_POWER:.1f} W")

Effective rate:  1.3133e+12 FLOPS
Processing time: 6.6265 ms
Compute time:    6.6246 ms
P_dynamic:       34.25 W
P_static:        9.0 W


In [3]:
frames = load_results("frames")
energy = load_results("energy")
completed = frames.filter(pl.col("event").is_in(["processed", "violated"]))

## 1. Compute time matches formula

$$T_{\text{compute}} = \frac{D}{R_{\text{peak}} \cdot \frac{f_{\text{min}}}{f_{\text{max}}} \cdot \eta}$$

In [4]:
compute_check = completed.select(
    pl.col("compute_ms").mean().alias("obs_mean"),
    pl.col("compute_ms").std().alias("obs_std"),
    pl.col("compute_ms").min().alias("obs_min"),
    pl.col("compute_ms").max().alias("obs_max"),
    pl.lit(T_PROC * 1e3).alias("expected"),
).with_columns(
    (pl.col("obs_mean") - pl.col("expected")).abs().alias("error_ms"),
)
print(compute_check)

shape: (1, 6)
┌──────────┬────────────┬─────────┬─────────┬──────────┬──────────┐
│ obs_mean ┆ obs_std    ┆ obs_min ┆ obs_max ┆ expected ┆ error_ms │
│ ---      ┆ ---        ┆ ---     ┆ ---     ┆ ---      ┆ ---      │
│ f64      ┆ f64        ┆ f64     ┆ f64     ┆ f64      ┆ f64      │
╞══════════╪════════════╪═════════╪═════════╪══════════╪══════════╡
│ 6.6265   ┆ 1.5299e-15 ┆ 6.6265  ┆ 6.6265  ┆ 6.626541 ┆ 0.000041 │
└──────────┴────────────┴─────────┴─────────┴──────────┴──────────┘


## 2. Energy matches formula

$$E = P_{\text{static}} \times T_{\text{sim}} + C_{\text{eff}} \times V^2 \times f \times \sum_i T_{\text{compute},i}$$

In [5]:
rows = []
for bid in sorted(energy["backend_id"].unique().to_list()):
    subset = energy.filter(pl.col("backend_id") == bid)
    obs = subset["total_energy_j"].mean()
    n_tasks = subset["frames_processed"].mean()

    exp = STATIC_POWER * TOTAL_TIME + P_DYN * n_tasks * T_COMPUTE

    rows.append({"backend_id": bid, "n_tasks": int(n_tasks),
                  "obs_J": round(obs, 4), "exp_J": round(exp, 4),
                  "error_J": round(abs(obs - exp), 4)})

energy_check = pl.DataFrame(rows)
print(energy_check)

shape: (2, 5)
┌────────────┬─────────┬──────────┬──────────┬─────────┐
│ backend_id ┆ n_tasks ┆ obs_J    ┆ exp_J    ┆ error_J │
│ ---        ┆ ---     ┆ ---      ┆ ---      ┆ ---     │
│ i64        ┆ i64     ┆ f64      ┆ f64      ┆ f64     │
╞════════════╪═════════╪══════════╪══════════╪═════════╡
│ 0          ┆ 569     ┆ 327.1148 ┆ 327.1148 ┆ 0.0     │
│ 1          ┆ 0       ┆ 198.0    ┆ 198.0    ┆ 0.0     │
└────────────┴─────────┴──────────┴──────────┴─────────┘
